## Undergraduate Research
# *Developing a neural network model that can estimate surface curvatures*
### This research was conducted under Dr. Jacob Hauenstein in the Department of Computer Science at the University of Alabama in Huntsville. This work aims to develop a neural network model capable of estimating surface curvatures from range image and/or function value data of those surfaces. We first trained our designed model on both range image data and function value data due to the limited availability of range image data. However, as these two types of data are not similar in nature, our model did not achieve good results. As a result, we focused on working with function value data only, as such data are easier to generate, and thus not as limited as range image data.
### However, further experimentation with function value data revealed that working with signed labels was not feasible, as the behavior of the data for different generated functions was significantly different even when the function types were similar in nature. So, we proceeded to work with unsigned labels, deferring the research on identifying the curvature signs to a later stage using a different model. Despite this adjustment, our model still failed to produce satisfactory performance, suggesting that a change in model architecture is necessary to achieve better results for our work.

## Training and evaluating a model on range image data

In [ ]:
import sympy
import numpy as np
import tensorflow as tf
from tensorflow import keras
from random import uniform
from sklearn.utils import shuffle
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Activation, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

In [2]:
# Function for extracting input data from raw image files

def data(fname, fsize_x, fsize_y):
    thedata = np.fromfile(fname, dtype="double", count=-1)
    thedata = thedata.reshape((fsize_x, fsize_y), order="F")
    
    mjlist = []
    mnlist = []
    
    for row in range(0, fsize_x-2):
        for col in range(0, fsize_y-2):
            for r in range(row, row+3):
                for c in range(col, col+3):
                    mnlist.append(thedata[r][c])
            mjlist.append(mnlist)
            mnlist = []
    
    t_samples = mjlist
    return t_samples

In [3]:
# Function for extracting labels from raw image files

def label(fname, fsize_x, fsize_y):
    thelabel = np.fromfile(fname, dtype="double", count=-1)
    thelabel = thelabel.reshape((fsize_x, fsize_y, 2), order="F")
    
    lblist = []
    temp_list = []
    
    for row in range(1, fsize_x-1):
        for col in range(1, fsize_y-1):
            mn, mx = thelabel[row][col]
            temp_list.extend([mn, mx])
            lblist.append(temp_list)
            temp_list = []
            
    t_labels = lblist
    return t_labels

In [ ]:
temp_samples = data("../files/ML256.raw", 256, 256)
temp_labels = label("../files/ML256solution.raw", 256, 256)

In [5]:
l_samples = temp_samples
l_labels = temp_labels

In [6]:
temp_samples = data("../files/hp512.raw", 512, 512)
temp_labels = label("../files/hp512solution.raw", 512, 512)

In [7]:
l_samples.extend(temp_samples)
l_labels.extend(temp_labels)

In [8]:
temp_samples = data("../files/cylinder128.raw", 128, 128)
temp_labels = label("../files/cylinder128solution.raw", 128, 128)

In [9]:
l_samples.extend(temp_samples)
l_labels.extend(temp_labels)

In [10]:
samples = np.array(l_samples)
labels = np.array(l_labels)

In [12]:
train_samples, test_samples_1, train_labels, test_labels_1 = train_test_split(samples, labels, test_size = 0.25, random_state = 0)

##### This model architecture was decided randomly, based on theoretical aspects and common features that have been proven successful. We would change the architecture if the results aren't up to the mark.

In [13]:
model = Sequential([
    Dense(units = 6, input_shape = (9,), activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 6, activation = 'relu'),
    Dense(units = 2)
])

In [14]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])
model.fit(x = train_samples, y = train_labels, validation_split = 0.25, batch_size = 20, epochs = 25, shuffle = True, verbose = 2)

Epoch 1/25
9577/9577 - 27s - loss: 644.1799 - mean_absolute_error: 2.5895 - val_loss: 1.3590 - val_mean_absolute_error: 0.9145 - 27s/epoch - 3ms/step
Epoch 2/25
9577/9577 - 23s - loss: 0.4242 - mean_absolute_error: 0.3675 - val_loss: 0.1339 - val_mean_absolute_error: 0.2825 - 23s/epoch - 2ms/step
Epoch 3/25
9577/9577 - 24s - loss: 0.0961 - mean_absolute_error: 0.1508 - val_loss: 0.0090 - val_mean_absolute_error: 0.0696 - 24s/epoch - 2ms/step
Epoch 4/25
9577/9577 - 24s - loss: 0.0039 - mean_absolute_error: 0.0293 - val_loss: 9.3783e-05 - val_mean_absolute_error: 0.0053 - 24s/epoch - 3ms/step
Epoch 5/25
9577/9577 - 28s - loss: 6.3944e-05 - mean_absolute_error: 0.0030 - val_loss: 2.8486e-05 - val_mean_absolute_error: 0.0025 - 28s/epoch - 3ms/step
Epoch 6/25
9577/9577 - 26s - loss: 4.1997e-05 - mean_absolute_error: 0.0027 - val_loss: 2.8233e-05 - val_mean_absolute_error: 0.0026 - 26s/epoch - 3ms/step
Epoch 7/25
9577/9577 - 24s - loss: 3.6677e-05 - mean_absolute_error: 0.0026 - val_loss: 5.

In [15]:
model.save('Research_2ndApproach.h5')

In [16]:
test_samples_1.shape

(85123, 9)

In [17]:
test_labels_1.shape

(85123, 2)

In [18]:
result = model.evaluate(x = test_samples_1, y = test_labels_1, batch_size = 20, verbose = 2)

4257/4257 - 4s - loss: 2.2357e-05 - mean_absolute_error: 0.0021 - 4s/epoch - 836us/step


### The results looked pretty decent. However, we have shortage of range image data that covers different types of surfaces. So, to generalize our model well, we focus on generating our own data for the research.

## Continuing to train our previously trained model on our generated function value data

In [2]:
x = sympy.Symbol('x')
y = sympy.Symbol('y')

In [3]:
# Function for generating input sample and unsigned label for a particular point of interest

def data_generator(exp, center_x, center_y):
    l_sample = []
    
    for i in range(center_x-1, center_x+2):
        for j in range(center_y-1, center_y+2):
            value = exp.evalf(subs={x: i, y: j})
            l_sample.append(value)
            
    fx = exp.diff(x, 1)
    fy = exp.diff(y, 1)
    fxx = exp.diff(x, 2)
    fyy = exp.diff(y, 2)
    fxy = exp.diff(x, y, 1)
    
    v_fx = fx.evalf(subs={x: center_x, y: center_y})
    v_fy = fy.evalf(subs={x: center_x, y: center_y})
    v_fxx = fxx.evalf(subs={x: center_x, y: center_y})
    v_fyy = fyy.evalf(subs={x: center_x, y: center_y})
    v_fxy = fxy.evalf(subs={x: center_x, y: center_y})
    
    K = (v_fxx*v_fyy - v_fxy**2) / (1 + v_fx**2 + v_fy**2)**2
    H = (v_fxx + v_fyy + v_fxx*v_fy**2 + v_fyy*v_fx**2 - 2*v_fx*v_fy*v_fxy) / (2 * (1 + v_fx**2 + v_fy**2)**1.5)
    
    k1 = H + (H**2 - K)**0.5
    k2 = H - (H**2 - K)**0.5
    
    #************************************************************************************************************
    # Start of the changes made for creating unsigned labels
    #************************************************************************************************************
    u_k1 = abs(k1)
    u_k2 = abs(k2)
    
    if(u_k1 < u_k2):
        temp = u_k1
        u_k1 = u_k2
        u_k2 = temp
    
    l_label = []
    l_label.extend([u_k1, u_k2])
    
    #************************************************************************************************************
    # End of the changes made for creating unsigned labels
    #************************************************************************************************************
    
    return l_sample, l_label

In [4]:
# Function for generating a list of samples and labels for a range of points

def datalist_generator(exp, x_start, y_start, x_end, y_end):
    l_samples = []
    l_labels = []

    for i in range(x_start, x_end+1):
        for j in range(y_start, y_end+1):
            t_sample, t_label = data_generator(exp, i, j)
            l_samples.append(t_sample)
            l_labels.append(t_label)
        
    samples = np.array(l_samples, dtype = 'float64')
    labels = np.array(l_labels, dtype = 'float64')
    
    return samples, labels

In [5]:
# Function for generating expressions

def exp_generator(num, h_deg):
    
    if (num == 1):
        a = uniform(-5.0, 5.0)
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-5.0, 5.0)
        e = uniform(-5.0, 5.0)
        
        if(h_deg == 3):
            a = 0.0
        elif(h_deg == 2):
            a = 0.0
            b = 0.0
        
        f = a*x**4 + b*x**3 + c*x**2 + d*x + e
        
        return f
    
    elif (num == 2):
        a = uniform(-5.0, 5.0)
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-5.0, 5.0)
        e = uniform(-5.0, 5.0)
        f = uniform(-5.0, 5.0)
        g = uniform(-5.0, 5.0)
        h = uniform(-5.0, 5.0)
        i = uniform(-5.0, 5.0)
        j = uniform(-5.0, 5.0)
        k = uniform(-5.0, 5.0)
        l = uniform(-5.0, 5.0)
        m = uniform(-5.0, 5.0)
        n = uniform(-5.0, 5.0)
        o = uniform(-5.0, 5.0)
        
        if(h_deg == 3):
            a = 0.0
            b = 0.0
            c = 0.0
            d = 0.0
            e = 0.0
        elif(h_deg == 2):
            a = 0.0
            b = 0.0
            c = 0.0
            d = 0.0
            e = 0.0
            f = 0.0
            g = 0.0
            h = 0.0
            i = 0.0
        
        f = a*x**4 + b*y**4 + c*x**3*y + d*x*y**3 + e*x**2*y**2 + f*x**3 + g*y**3 + h*x**2*y + i*x*y**2 + j*x**2 + k*y**2 + l*x*y + m*x + n*y + o
        
        return f
    
    elif (num == 3):
        a = uniform(-5.0, 5.0)
        
        while(a == 0):
            a = uniform(-5.0, 5.0)
        
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-5.0, 5.0)
        
        f = a*sympy.sin(b*x-c) + d
        
        return f
    
    elif (num == 4):
        a = uniform(-5.0, 5.0)
        
        while(a == 0):
            a = uniform(-5.0, 5.0)
        
        b = uniform(-5.0, 5.0)
        c = uniform(-5.0, 5.0)
        d = uniform(-5.0, 5.0)
        
        f = a*sympy.cos(b*x-c) + d
        
        return f
    
    else:
        f = 0
        return f

In [7]:
funct = exp_generator(1, 2)

In [8]:
funct

-2.13283088951388*x**2 + 4.54798819329558*x + 0.136493748324478

In [9]:
s, l = datalist_generator(funct, 1, 1, 254, 254)

In [10]:
s.shape

(64516, 9)

In [11]:
l.shape

(64516, 2)

In [12]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [13]:
from tensorflow.keras.models import load_model
model = load_model('Research_2ndApproach.h5')

In [14]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 25, shuffle = True, verbose = 2)

Epoch 1/25
1694/1694 - 2s - loss: 91.2030 - mean_absolute_error: 0.9697 - val_loss: 0.0450 - val_mean_absolute_error: 0.1302 - 2s/epoch - 1ms/step
Epoch 2/25
1694/1694 - 2s - loss: 0.0354 - mean_absolute_error: 0.0724 - val_loss: 0.0266 - val_mean_absolute_error: 0.0209 - 2s/epoch - 932us/step
Epoch 3/25
1694/1694 - 2s - loss: 0.1591 - mean_absolute_error: 0.1155 - val_loss: 1.1083 - val_mean_absolute_error: 0.7708 - 2s/epoch - 1ms/step
Epoch 4/25
1694/1694 - 2s - loss: 0.4274 - mean_absolute_error: 0.2298 - val_loss: 0.0261 - val_mean_absolute_error: 0.0392 - 2s/epoch - 1ms/step
Epoch 5/25
1694/1694 - 2s - loss: 0.2125 - mean_absolute_error: 0.1044 - val_loss: 0.2630 - val_mean_absolute_error: 0.3696 - 2s/epoch - 1ms/step
Epoch 6/25
1694/1694 - 2s - loss: 0.8390 - mean_absolute_error: 0.2237 - val_loss: 0.0173 - val_mean_absolute_error: 0.0393 - 2s/epoch - 1ms/step
Epoch 7/25
1694/1694 - 2s - loss: 0.2461 - mean_absolute_error: 0.1854 - val_loss: 3.7065 - val_mean_absolute_error: 1.41

In [15]:
model.save('Research_2ndApproach.h5')

##### Model is evaluated after each training to check overfitting

In [16]:
result = model.evaluate(x = test_samples, y = test_labels, batch_size = 20, verbose = 2)

968/968 - 1s - loss: 0.0072 - mean_absolute_error: 0.0103 - 1s/epoch - 1ms/step


In [7]:
funct = exp_generator(1, 3)

In [8]:
funct

-4.1480891217579*x**3 + 3.64567656664299*x**2 - 0.501819908897123*x + 1.75043476348186

In [9]:
s, l = datalist_generator(funct, 20, 20, 273, 273)

In [10]:
s.shape

(64516, 9)

In [11]:
l.shape

(64516, 2)

In [12]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [14]:
from tensorflow.keras.models import load_model
model = load_model('Research_2ndApproach.h5')

In [15]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 25, shuffle = True, verbose = 2)

Epoch 1/25
1694/1694 - 4s - loss: 1.9119e-06 - mean_absolute_error: 6.0274e-04 - val_loss: 2.6525e-15 - val_mean_absolute_error: 3.6504e-08 - 4s/epoch - 3ms/step
Epoch 2/25
1694/1694 - 5s - loss: 7.4585e-17 - mean_absolute_error: 2.0671e-09 - val_loss: 9.2465e-20 - val_mean_absolute_error: 7.8121e-11 - 5s/epoch - 3ms/step
Epoch 3/25
1694/1694 - 3s - loss: 9.8097e-20 - mean_absolute_error: 8.1270e-11 - val_loss: 9.2502e-20 - val_mean_absolute_error: 7.6050e-11 - 3s/epoch - 2ms/step
Epoch 4/25
1694/1694 - 4s - loss: 9.8364e-20 - mean_absolute_error: 8.1438e-11 - val_loss: 9.2558e-20 - val_mean_absolute_error: 7.4182e-11 - 4s/epoch - 2ms/step
Epoch 5/25
1694/1694 - 4s - loss: 9.8690e-20 - mean_absolute_error: 8.1515e-11 - val_loss: 9.2487e-20 - val_mean_absolute_error: 7.6751e-11 - 4s/epoch - 2ms/step
Epoch 6/25
1694/1694 - 4s - loss: 9.9807e-20 - mean_absolute_error: 8.2711e-11 - val_loss: 1.0847e-19 - val_mean_absolute_error: 1.5589e-10 - 4s/epoch - 3ms/step
Epoch 7/25
1694/1694 - 4s - 

In [16]:
model.save('Research_2ndApproach.h5')

In [17]:
result = model.evaluate(x = test_samples, y = test_labels, batch_size = 20, verbose = 2)

968/968 - 1s - loss: 1.2569e-14 - mean_absolute_error: 7.9273e-08 - 1s/epoch - 1ms/step


In [7]:
funct = exp_generator(1, 4)

In [8]:
funct

-0.589132276222559*x**4 - 2.81321822714685*x**3 - 0.207598296034716*x**2 + 2.34367235892218*x + 3.46752165705189

In [9]:
s, l = datalist_generator(funct, 1, 1, 254, 254)

In [10]:
s.shape

(64516, 9)

In [11]:
l.shape

(64516, 2)

In [12]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [13]:
from tensorflow.keras.models import load_model
model = load_model('Research_2ndApproach.h5')

In [14]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 25, shuffle = True, verbose = 2)

Epoch 1/25
1694/1694 - 7s - loss: 6.0237e-05 - mean_absolute_error: 2.7623e-04 - val_loss: 3.0051e-07 - val_mean_absolute_error: 6.1589e-05 - 7s/epoch - 4ms/step
Epoch 2/25
1694/1694 - 6s - loss: 2.6572e-07 - mean_absolute_error: 6.6601e-05 - val_loss: 2.2936e-07 - val_mean_absolute_error: 5.3364e-05 - 6s/epoch - 4ms/step
Epoch 3/25
1694/1694 - 6s - loss: 2.2657e-06 - mean_absolute_error: 1.2825e-04 - val_loss: 1.8145e-07 - val_mean_absolute_error: 4.5570e-05 - 6s/epoch - 4ms/step
Epoch 4/25
1694/1694 - 6s - loss: 1.1852e-07 - mean_absolute_error: 5.0173e-05 - val_loss: 8.8750e-08 - val_mean_absolute_error: 6.6719e-05 - 6s/epoch - 4ms/step
Epoch 5/25
1694/1694 - 6s - loss: 9.6089e-07 - mean_absolute_error: 1.2072e-04 - val_loss: 7.8356e-08 - val_mean_absolute_error: 8.6640e-05 - 6s/epoch - 4ms/step
Epoch 6/25
1694/1694 - 6s - loss: 1.2032e-06 - mean_absolute_error: 1.1649e-04 - val_loss: 2.9158e-08 - val_mean_absolute_error: 2.2629e-05 - 6s/epoch - 4ms/step
Epoch 7/25
1694/1694 - 6s - 

###### Results were continuously oscillating

In [15]:
model.save('Research_2ndApproach.h5')

In [16]:
result = model.evaluate(x = test_samples, y = test_labels, batch_size = 20, verbose = 2)

968/968 - 2s - loss: 4.8426e-09 - mean_absolute_error: 9.7564e-06 - 2s/epoch - 2ms/step


In [7]:
funct = exp_generator(2, 2)

In [8]:
funct

4.35347405160217*x**2 + 0.101591594260528*x*y - 1.26254719501396*x + 3.32002727897141*y**2 - 3.48053535296255*y + 1.71019073841434

In [9]:
s, l = datalist_generator(funct, 25, 25, 278, 278)

In [10]:
s.shape

(64516, 9)

In [11]:
l.shape

(64516, 2)

In [12]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [13]:
from tensorflow.keras.models import load_model
model = load_model('Research_2ndApproach.h5')

In [14]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 25, shuffle = True, verbose = 2)

Epoch 1/25
1694/1694 - 6s - loss: 204062624.0000 - mean_absolute_error: 1353.5392 - val_loss: 0.0057 - val_mean_absolute_error: 0.0331 - 6s/epoch - 3ms/step
Epoch 2/25
1694/1694 - 6s - loss: 0.0059 - mean_absolute_error: 0.0309 - val_loss: 0.0054 - val_mean_absolute_error: 0.0294 - 6s/epoch - 3ms/step
Epoch 3/25
1694/1694 - 6s - loss: 0.0058 - mean_absolute_error: 0.0296 - val_loss: 0.0054 - val_mean_absolute_error: 0.0292 - 6s/epoch - 4ms/step
Epoch 4/25
1694/1694 - 4s - loss: 0.0058 - mean_absolute_error: 0.0295 - val_loss: 0.0054 - val_mean_absolute_error: 0.0292 - 4s/epoch - 3ms/step
Epoch 5/25
1694/1694 - 6s - loss: 0.0058 - mean_absolute_error: 0.0296 - val_loss: 0.0054 - val_mean_absolute_error: 0.0295 - 6s/epoch - 4ms/step
Epoch 6/25
1694/1694 - 5s - loss: 0.0059 - mean_absolute_error: 0.0305 - val_loss: 0.0054 - val_mean_absolute_error: 0.0298 - 5s/epoch - 3ms/step
Epoch 7/25
1694/1694 - 5s - loss: 5830.4976 - mean_absolute_error: 11.1317 - val_loss: 1.7869 - val_mean_absolute

In [15]:
model.save('Research_2ndApproach_temp.h5')

In [16]:
result = model.evaluate(x = test_samples, y = test_labels, batch_size = 20, verbose = 2)

968/968 - 1s - loss: 0.0076 - mean_absolute_error: 0.0646 - 1s/epoch - 1ms/step


In [2]:
from tensorflow.keras.models import load_model
model = load_model('Research_2ndApproach_temp.h5')

In [3]:
model.save('Research_2ndApproach.h5') ## Eventually deleted, since we later focused on function values only.

### The results in this experiment eventually were quite worse, prompting us to continue with function value data only, as the range image data and generated function value data differs in type.

## Training and testing with samples and labels generated from created expressions (function values) from scratch

In [7]:
model = Sequential([
    Dense(units = 6, input_shape = (9,), activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 6, activation = 'relu'),
    Dense(units = 2)
])

In [8]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [9]:
funct = exp_generator(1, 2)

In [10]:
funct

2.33535371680227*x**2 + 0.385356840375302*x + 3.91546549510469

In [11]:
s, l = datalist_generator(funct, 1, 1, 254, 254)

In [12]:
s.shape

(64516, 9)

In [13]:
l.shape

(64516, 2)

In [14]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [15]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 25, shuffle = True, verbose = 2)

Epoch 1/25
1694/1694 - 3s - loss: 1876893.0000 - mean_absolute_error: 486.5560 - val_loss: 10.6587 - val_mean_absolute_error: 2.3755 - 3s/epoch - 2ms/step
Epoch 2/25
1694/1694 - 2s - loss: 0.1058 - mean_absolute_error: 0.0649 - val_loss: 0.0015 - val_mean_absolute_error: 0.0366 - 2s/epoch - 1ms/step
Epoch 3/25
1694/1694 - 2s - loss: 0.0015 - mean_absolute_error: 0.0365 - val_loss: 0.0015 - val_mean_absolute_error: 0.0365 - 2s/epoch - 1ms/step
Epoch 4/25
1694/1694 - 2s - loss: 0.0015 - mean_absolute_error: 0.0364 - val_loss: 0.0015 - val_mean_absolute_error: 0.0363 - 2s/epoch - 1ms/step
Epoch 5/25
1694/1694 - 2s - loss: 0.0015 - mean_absolute_error: 0.0362 - val_loss: 0.0015 - val_mean_absolute_error: 0.0360 - 2s/epoch - 1ms/step
Epoch 6/25
1694/1694 - 2s - loss: 0.0014 - mean_absolute_error: 0.0357 - val_loss: 0.0014 - val_mean_absolute_error: 0.0353 - 2s/epoch - 1ms/step
Epoch 7/25
1694/1694 - 2s - loss: 0.0014 - mean_absolute_error: 0.0349 - val_loss: 0.0013 - val_mean_absolute_error

In [16]:
model.save('New_Research_2ndApproach.h5')

In [17]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

968/968 - 1s - loss: 2.1174e-06 - mean_absolute_error: 1.9895e-04 - 674ms/epoch - 696us/step


In [7]:
funct = exp_generator(1, 3)

In [8]:
funct

-2.75496051263759*x**3 + 1.12639535310838*x**2 - 1.8823838932756*x - 2.61955856901084

In [9]:
s, l = datalist_generator(funct, 20, 20, 273, 273)

In [10]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [11]:
from tensorflow.keras.models import load_model
model = load_model('New_Research_2ndApproach.h5')

In [12]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 25, shuffle = True, verbose = 2)

Epoch 1/25
1694/1694 - 2s - loss: 653712896.0000 - mean_absolute_error: 2461.0408 - val_loss: 15991.5557 - val_mean_absolute_error: 72.1577 - 2s/epoch - 1ms/step
Epoch 2/25
1694/1694 - 2s - loss: 15121.1016 - mean_absolute_error: 72.5453 - val_loss: 14052.4170 - val_mean_absolute_error: 71.2840 - 2s/epoch - 1ms/step
Epoch 3/25
1694/1694 - 2s - loss: 12388.2070 - mean_absolute_error: 63.5448 - val_loss: 596.9546 - val_mean_absolute_error: 14.2073 - 2s/epoch - 1ms/step
Epoch 4/25
1694/1694 - 2s - loss: 6981.9106 - mean_absolute_error: 23.7070 - val_loss: 48.9491 - val_mean_absolute_error: 3.9713 - 2s/epoch - 1ms/step
Epoch 5/25
1694/1694 - 2s - loss: 101589.9297 - mean_absolute_error: 103.2659 - val_loss: 28.9018 - val_mean_absolute_error: 3.1570 - 2s/epoch - 998us/step
Epoch 6/25
1694/1694 - 2s - loss: 133923.1250 - mean_absolute_error: 66.0490 - val_loss: 107014.6641 - val_mean_absolute_error: 230.2607 - 2s/epoch - 1ms/step
Epoch 7/25
1694/1694 - 2s - loss: 303460.1250 - mean_absolute_

##### The results look pretty bad!!!

In [13]:
model.save('New_Research_2ndApproach.h5')

In [14]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

968/968 - 1s - loss: 1.0497 - mean_absolute_error: 0.5105 - 655ms/epoch - 677us/step


In [7]:
funct = exp_generator(1, 4)

In [8]:
funct

4.86062152627121*x**4 - 1.03817120161985*x**3 - 3.87086872742696*x**2 - 0.236580730997042*x + 0.0143534642649543

In [9]:
s, l = datalist_generator(funct, 1, 1, 254, 254)

In [10]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [11]:
from tensorflow.keras.models import load_model
model = load_model('New_Research_2ndApproach.h5')

In [12]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 25, shuffle = True, verbose = 2)

Epoch 1/25
1694/1694 - 8s - loss: 283291877376.0000 - mean_absolute_error: 15173.2773 - val_loss: 0.0055 - val_mean_absolute_error: 0.0313 - 8s/epoch - 5ms/step
Epoch 2/25
1694/1694 - 6s - loss: 0.0025 - mean_absolute_error: 0.0284 - val_loss: 0.0015 - val_mean_absolute_error: 0.0273 - 6s/epoch - 4ms/step
Epoch 3/25
1694/1694 - 6s - loss: 0.0011 - mean_absolute_error: 0.0267 - val_loss: 8.8770e-04 - val_mean_absolute_error: 0.0264 - 6s/epoch - 4ms/step
Epoch 4/25
1694/1694 - 6s - loss: 8.0151e-04 - mean_absolute_error: 0.0261 - val_loss: 7.5650e-04 - val_mean_absolute_error: 0.0258 - 6s/epoch - 4ms/step
Epoch 5/25
1694/1694 - 7s - loss: 7.1388e-04 - mean_absolute_error: 0.0256 - val_loss: 6.8682e-04 - val_mean_absolute_error: 0.0254 - 7s/epoch - 4ms/step
Epoch 6/25
1694/1694 - 6s - loss: 6.8269e-04 - mean_absolute_error: 0.0254 - val_loss: 6.7856e-04 - val_mean_absolute_error: 0.0254 - 6s/epoch - 3ms/step
Epoch 7/25
1694/1694 - 6s - loss: 6.8312e-04 - mean_absolute_error: 0.0254 - val_

##### The model again works better for univariate quartic values.

In [13]:
model.save('New_Research_2ndApproach.h5')

## Deleted, as this model was trained on a dataset with signed labels, but we eventually worked with unsigned labels.

In [14]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

968/968 - 2s - loss: 2.3294e-11 - mean_absolute_error: 4.4456e-07 - 2s/epoch - 2ms/step


In [7]:
funct = exp_generator(1, 2)

In [8]:
funct

4.47449572207884*x**2 + 0.125076925503204*x - 2.33756730720867

In [9]:
s, l = datalist_generator(funct, 1, 1, 254, 254)

In [10]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [11]:
from tensorflow.keras.models import load_model
model = load_model('New_Research_2ndApproach.h5')

In [12]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

968/968 - 2s - loss: 6.1192e-06 - mean_absolute_error: 3.5610e-04 - 2s/epoch - 2ms/step


### The model retains the training on univariate quadratic function values; however, its failure to estimate the curvatures for univariate cubic functions was concerning!!

## Training and testing with a smaller dataset, generated from created expressions from scratch

### This is done to guage the behavior of our model on different types of functions

In [7]:
funct = exp_generator(1, 2)

In [8]:
funct

-3.81846491229682*x**2 - 3.38807849718933*x + 0.681770394538336

In [9]:
s, l = datalist_generator(funct, 25, 25, 38, 38)

In [10]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [11]:
model = Sequential([
    Dense(units = 6, input_shape = (9,), activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 6, activation = 'relu'),
    Dense(units = 2)
])

In [12]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [13]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 25, shuffle = True, verbose = 2)

Epoch 1/25
6/6 - 3s - loss: 3.3962e-10 - mean_absolute_error: 9.6957e-06 - val_loss: 4.2167e-12 - val_mean_absolute_error: 1.4472e-06 - 3s/epoch - 471ms/step
Epoch 2/25
6/6 - 0s - loss: 1.5715e-10 - mean_absolute_error: 7.4734e-06 - val_loss: 2.3858e-10 - val_mean_absolute_error: 1.0921e-05 - 96ms/epoch - 16ms/step
Epoch 3/25
6/6 - 0s - loss: 1.2003e-10 - mean_absolute_error: 7.1944e-06 - val_loss: 1.0516e-11 - val_mean_absolute_error: 2.2900e-06 - 89ms/epoch - 15ms/step
Epoch 4/25
6/6 - 0s - loss: 4.7553e-11 - mean_absolute_error: 4.3492e-06 - val_loss: 6.6107e-11 - val_mean_absolute_error: 5.7480e-06 - 87ms/epoch - 14ms/step
Epoch 5/25
6/6 - 0s - loss: 3.3903e-11 - mean_absolute_error: 3.8412e-06 - val_loss: 8.0263e-12 - val_mean_absolute_error: 1.9998e-06 - 87ms/epoch - 14ms/step
Epoch 6/25
6/6 - 0s - loss: 1.5551e-11 - mean_absolute_error: 2.5120e-06 - val_loss: 1.0579e-11 - val_mean_absolute_error: 2.2968e-06 - 85ms/epoch - 14ms/step
Epoch 7/25
6/6 - 0s - loss: 8.0370e-12 - mean_a

In [14]:
model.save('Smaller_Research_2ndApproach.h5')

In [15]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

3/3 - 0s - loss: 2.9659e-14 - mean_absolute_error: 1.0278e-07 - 51ms/epoch - 17ms/step


In [16]:
funct = exp_generator(1, 3)

In [17]:
funct

3.68098107081366*x**3 - 4.94965059488966*x**2 - 3.44973198080478*x - 3.29116295257791

In [18]:
s, l = datalist_generator(funct, 1, 1, 14, 14)

In [19]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [20]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 25, shuffle = True, verbose = 2)

Epoch 1/25
6/6 - 0s - loss: 32638.8555 - mean_absolute_error: 106.2702 - val_loss: 34464.5820 - val_mean_absolute_error: 121.1047 - 145ms/epoch - 24ms/step
Epoch 2/25
6/6 - 0s - loss: 25443.7422 - mean_absolute_error: 94.5932 - val_loss: 25896.3887 - val_mean_absolute_error: 106.4557 - 95ms/epoch - 16ms/step
Epoch 3/25
6/6 - 0s - loss: 19362.9844 - mean_absolute_error: 83.4798 - val_loss: 19757.3457 - val_mean_absolute_error: 94.2176 - 85ms/epoch - 14ms/step
Epoch 4/25
6/6 - 0s - loss: 14890.7451 - mean_absolute_error: 74.3527 - val_loss: 15460.4219 - val_mean_absolute_error: 84.5346 - 85ms/epoch - 14ms/step
Epoch 5/25
6/6 - 0s - loss: 11567.2725 - mean_absolute_error: 66.5270 - val_loss: 11979.6855 - val_mean_absolute_error: 75.6248 - 81ms/epoch - 13ms/step
Epoch 6/25
6/6 - 0s - loss: 8693.6455 - mean_absolute_error: 58.9830 - val_loss: 9014.0508 - val_mean_absolute_error: 66.7108 - 86ms/epoch - 14ms/step
Epoch 7/25
6/6 - 0s - loss: 6604.8750 - mean_absolute_error: 52.2459 - val_loss:

In [21]:
model.save('Smaller_Research_2ndApproach.h5')

##### The model still behaves poorly, even with smaller number of samples, indicating its inability to identify patterns with univariate cubic functions.

In [22]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

3/3 - 0s - loss: 10.3275 - mean_absolute_error: 2.1533 - 46ms/epoch - 15ms/step


In [7]:
funct = exp_generator(1, 3)

In [8]:
funct

4.51720057160143*x**3 - 3.73252745980195*x**2 + 0.129458361963558*x - 1.51129232076624

In [9]:
s, l = datalist_generator(funct, 1, 1, 14, 14)

In [10]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [11]:
from tensorflow.keras.models import load_model
model = load_model('Smaller_Research_2ndApproach.h5')

In [12]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 35, shuffle = True, verbose = 2)

Epoch 1/35
6/6 - 2s - loss: 14.5492 - mean_absolute_error: 2.5884 - val_loss: 11.6232 - val_mean_absolute_error: 2.1726 - 2s/epoch - 342ms/step
Epoch 2/35
6/6 - 0s - loss: 12.4209 - mean_absolute_error: 2.3372 - val_loss: 10.1306 - val_mean_absolute_error: 1.9990 - 102ms/epoch - 17ms/step
Epoch 3/35
6/6 - 0s - loss: 10.5047 - mean_absolute_error: 2.1197 - val_loss: 9.0373 - val_mean_absolute_error: 1.8472 - 83ms/epoch - 14ms/step
Epoch 4/35
6/6 - 0s - loss: 9.1425 - mean_absolute_error: 1.9480 - val_loss: 8.0591 - val_mean_absolute_error: 1.7186 - 91ms/epoch - 15ms/step
Epoch 5/35
6/6 - 0s - loss: 8.0926 - mean_absolute_error: 1.7949 - val_loss: 7.2478 - val_mean_absolute_error: 1.6086 - 86ms/epoch - 14ms/step
Epoch 6/35
6/6 - 0s - loss: 7.2511 - mean_absolute_error: 1.6810 - val_loss: 6.5984 - val_mean_absolute_error: 1.5346 - 91ms/epoch - 15ms/step
Epoch 7/35
6/6 - 0s - loss: 6.4808 - mean_absolute_error: 1.5822 - val_loss: 6.0360 - val_mean_absolute_error: 1.4659 - 86ms/epoch - 14ms

In [13]:
model.save('Smaller_Research_2ndApproach.h5')

##### Training for more epochs slightly improved the results.

In [14]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

3/3 - 0s - loss: 2.0100 - mean_absolute_error: 0.8079 - 52ms/epoch - 17ms/step


In [15]:
funct = exp_generator(1, 4)

In [16]:
funct

2.65626461991613*x**4 - 2.82255165414781*x**3 + 0.723522664145423*x**2 - 3.42705702654563*x + 4.42659346367724

In [17]:
s, l = datalist_generator(funct, 12, 12, 25, 25)

In [18]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [19]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 25, shuffle = True, verbose = 2)

Epoch 1/25
6/6 - 0s - loss: 2.3200e-04 - mean_absolute_error: 0.0150 - val_loss: 2.3223e-04 - val_mean_absolute_error: 0.0150 - 145ms/epoch - 24ms/step
Epoch 2/25
6/6 - 0s - loss: 2.3230e-04 - mean_absolute_error: 0.0150 - val_loss: 2.3242e-04 - val_mean_absolute_error: 0.0150 - 87ms/epoch - 14ms/step
Epoch 3/25
6/6 - 0s - loss: 2.3246e-04 - mean_absolute_error: 0.0150 - val_loss: 2.3251e-04 - val_mean_absolute_error: 0.0150 - 85ms/epoch - 14ms/step
Epoch 4/25
6/6 - 0s - loss: 2.3253e-04 - mean_absolute_error: 0.0150 - val_loss: 2.3256e-04 - val_mean_absolute_error: 0.0150 - 83ms/epoch - 14ms/step
Epoch 5/25
6/6 - 0s - loss: 2.3257e-04 - mean_absolute_error: 0.0150 - val_loss: 2.3258e-04 - val_mean_absolute_error: 0.0150 - 80ms/epoch - 13ms/step
Epoch 6/25
6/6 - 0s - loss: 2.3258e-04 - mean_absolute_error: 0.0150 - val_loss: 2.3258e-04 - val_mean_absolute_error: 0.0150 - 85ms/epoch - 14ms/step
Epoch 7/25
6/6 - 0s - loss: 2.3258e-04 - mean_absolute_error: 0.0150 - val_loss: 2.3257e-04 -

In [20]:
model.save('Smaller_Research_2ndApproach.h5') 

## Deleted, as model trained on a smaller dataset won't be useful to us eventually, and this was only trained to understand the
## behavior of our generated data. Also, eventually we worked with unsigned labels only.

In [21]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

3/3 - 0s - loss: 2.3231e-04 - mean_absolute_error: 0.0150 - 52ms/epoch - 17ms/step


### Since the model is struggling with univariate cubic function values, this indicates that there could be significant differences between the cubic values and the other types of univariate function values, or the model is inadequate in identifying the different patterns for different functions. We first carefully study the data we are working on.

## Checking the labels list for different polynomials

In [7]:
funct1 = exp_generator(1, 2)

In [8]:
funct1

-4.84894178380111*x**2 - 4.88831854127242*x - 3.14861156604311

In [9]:
s1, l1 = datalist_generator(funct1, 36, 36, 50, 50)

In [10]:
train_samples1, test_samples1, train_labels1, test_labels1 = train_test_split(s1, l1, test_size = 0.3)

In [11]:
for i in train_labels1:
    print(i)

[ 0.00000000e+00 -1.38468496e-07]
[ 0.00000000e+00 -2.18583487e-07]
[ 0.00000000e+00 -1.60009645e-07]
[ 0.00000000e+00 -2.01560895e-07]
[ 0.00000000e+00 -1.05723187e-07]
[ 0.0000000e+00 -8.7643715e-08]
[ 0.00000000e+00 -1.86260948e-07]
[ 0.00000000e+00 -1.60009645e-07]
[ 0.00000000e+00 -1.72471135e-07]
[ 0.00000000e+00 -1.48720313e-07]
[ 0.00000000e+00 -1.86260948e-07]
[ 0.00000000e+00 -1.12847586e-07]
[ 0.00000000e+00 -9.31770245e-08]
[ 0.0000000e+00 -8.7643715e-08]
[ 0.00000000e+00 -1.29137695e-07]
[ 0.00000000e+00 -1.60009645e-07]
[ 0.00000000e+00 -1.48720313e-07]
[ 0.00000000e+00 -1.05723187e-07]
[ 0.00000000e+00 -1.60009645e-07]
[ 0.00000000e+00 -1.38468496e-07]
[ 0.00000000e+00 -1.12847586e-07]
[ 0.00000000e+00 -1.05723187e-07]
[ 0.00000000e+00 -1.29137695e-07]
[ 0.00000000e+00 -1.86260948e-07]
[ 0.00000000e+00 -8.25400005e-08]
[ 0.0000000e+00 -1.2062676e-07]
[ 0.00000000e+00 -9.31770245e-08]
[ 0.00000000e+00 -1.38468496e-07]
[ 0.00000000e+00 -9.31770245e-08]
[ 0.00000000e+00 -1.

In [ ]:
for i in test_labels1:
    print(i)

In [13]:
funct2 = exp_generator(1, 3)

In [14]:
funct2

3.12353899411423*x**3 + 1.38834494252503*x**2 - 4.08593480558979*x - 2.10314424520756

In [15]:
s2, l2 = datalist_generator(funct2, 36, 36, 50, 50)

In [16]:
train_samples2, test_samples2, train_labels2, test_labels2 = train_test_split(s2, l2, test_size = 0.3)

In [17]:
for i in train_labels2:
    print(i)

[1.08893491e-10 0.00000000e+00]
[2.47928404e-10 0.00000000e+00]
[1.71370668e-10 0.00000000e+00]
[1.52406255e-10 0.00000000e+00]
[1.93238661e-10 0.00000000e+00]
[7.94702138e-11 0.00000000e+00]
[1.08893491e-10 0.00000000e+00]
[3.22284625e-10 0.00000000e+00]
[1.93238661e-10 0.00000000e+00]
[7.94702138e-11 0.00000000e+00]
[2.18542379e-10 0.00000000e+00]
[9.7822411e-11 0.0000000e+00]
[3.69419154e-10 0.00000000e+00]
[1.93238661e-10 0.00000000e+00]
[2.18542379e-10 0.00000000e+00]
[1.71370668e-10 0.00000000e+00]
[3.69419154e-10 0.00000000e+00]
[1.35905426e-10 0.00000000e+00]
[7.94702138e-11 0.00000000e+00]
[3.22284625e-10 0.00000000e+00]
[3.22284625e-10 0.00000000e+00]
[2.18542379e-10 0.00000000e+00]
[1.52406255e-10 0.00000000e+00]
[7.18548237e-11 0.00000000e+00]
[3.22284625e-10 0.00000000e+00]
[9.7822411e-11 0.0000000e+00]
[1.52406255e-10 0.00000000e+00]
[1.2150276e-10 0.0000000e+00]
[1.52406255e-10 0.00000000e+00]
[2.82186197e-10 0.00000000e+00]
[3.22284625e-10 0.00000000e+00]
[1.71370668e-1

In [ ]:
for i in test_labels2:
    print(i)

In [19]:
funct3 = exp_generator(1, 4)

In [20]:
funct3

-4.25860098215478*x**4 - 4.614180326317*x**3 + 2.12044814687386*x**2 + 2.30782619100982*x + 1.36046699058469

In [21]:
s3, l3 = datalist_generator(funct3, 36, 36, 50, 50)

In [22]:
train_samples3, test_samples3, train_labels3, test_labels3 = train_test_split(s3, l3, test_size = 0.3)

In [23]:
for i in train_labels3:
    print(i)

[ 0.00000000e+00 -1.69396614e-14]
[ 0.00000000e+00 -5.07319096e-14]
[ 0.0000000e+00 -2.2780727e-14]
[ 0.00000000e+00 -2.65460988e-14]
[ 0.00000000e+00 -1.25306904e-13]
[ 0.00000000e+00 -1.25306904e-13]
[ 0.00000000e+00 -4.29025719e-14]
[ 0.00000000e+00 -1.46743748e-14]
[ 0.00000000e+00 -1.69396614e-14]
[ 0.00000000e+00 -2.65460988e-14]
[ 0.00000000e+00 -7.18333817e-14]
[ 0.00000000e+00 -8.60518718e-14]
[ 0.00000000e+00 -4.29025719e-14]
[ 0.00000000e+00 -7.18333817e-14]
[ 0.00000000e+00 -1.96135148e-14]
[ 0.00000000e+00 -4.29025719e-14]
[ 0.0000000e+00 -2.2780727e-14]
[ 0.00000000e+00 -3.64240575e-14]
[ 0.0000000e+00 -1.0357911e-13]
[ 0.00000000e+00 -1.96135148e-14]
[ 0.0000000e+00 -1.0357911e-13]
[ 0.00000000e+00 -1.46743748e-14]
[ 0.00000000e+00 -2.65460988e-14]
[ 0.0000000e+00 -1.0357911e-13]
[ 0.00000000e+00 -1.96135148e-14]
[ 0.00000000e+00 -1.25306904e-13]
[ 0.00000000e+00 -3.10397582e-14]
[ 0.00000000e+00 -5.07319096e-14]
[ 0.00000000e+00 -3.64240575e-14]
[ 0.0000000e+00 -1.03579

In [ ]:
for i in test_labels3:
    print(i)

### After observing the labels for different polynomials, it is identified that there are differences between the behavior of the curvature values for cubic and the other types of univariate polynomials because of using signed labels. Univariate polynomials have a singular curvature. As a result of using signed labels, cubic functions have a positive major curvature and the others have a negative minor curvature. That is why, the model failed to work with all types of functions. So next, we decided to check the behavior of our model on unsigned labels.

## Training and testing with a smaller dataset with "unsigned" labels from scratch

### We started with few no of samples to check whether the previous issue was solved just by using unsigned labels.

In [6]:
funct = exp_generator(1, 2)

In [7]:
funct

0.0953607378680772*x**2 + 4.34373026110564*x + 2.23307004349773

In [12]:
s, l = datalist_generator(funct, 31, 31, 44, 44)

In [13]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [16]:
model = Sequential([
    Dense(units = 6, input_shape = (9,), activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 6, activation = 'relu'),
    Dense(units = 2)
])

In [17]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [18]:
# Had to run this training multiple times from scratch to get these results

model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
6/6 - 3s - loss: 3.6435e-09 - mean_absolute_error: 3.6765e-05 - val_loss: 4.1145e-10 - val_mean_absolute_error: 1.3045e-05 - 3s/epoch - 467ms/step
Epoch 2/30
6/6 - 0s - loss: 1.5536e-09 - mean_absolute_error: 2.3439e-05 - val_loss: 7.3160e-10 - val_mean_absolute_error: 1.3968e-05 - 92ms/epoch - 15ms/step
Epoch 3/30
6/6 - 0s - loss: 6.2245e-10 - mean_absolute_error: 1.4540e-05 - val_loss: 7.2001e-10 - val_mean_absolute_error: 1.6759e-05 - 84ms/epoch - 14ms/step
Epoch 4/30
6/6 - 0s - loss: 5.2900e-10 - mean_absolute_error: 1.3421e-05 - val_loss: 9.9639e-10 - val_mean_absolute_error: 1.7647e-05 - 88ms/epoch - 15ms/step
Epoch 5/30
6/6 - 0s - loss: 4.8592e-10 - mean_absolute_error: 1.2755e-05 - val_loss: 6.6440e-10 - val_mean_absolute_error: 1.6160e-05 - 81ms/epoch - 13ms/step
Epoch 6/30
6/6 - 0s - loss: 4.7817e-10 - mean_absolute_error: 1.3054e-05 - val_loss: 5.3644e-10 - val_mean_absolute_error: 1.2051e-05 - 85ms/epoch - 14ms/step
Epoch 7/30
6/6 - 0s - loss: 3.5519e-10 - mean_a

In [19]:
model.save('Unsigned_Research_main.h5')

In [20]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

3/3 - 0s - loss: 3.4254e-10 - mean_absolute_error: 1.1031e-05 - 49ms/epoch - 16ms/step


In [6]:
funct = exp_generator(1, 3)

In [7]:
funct

3.95512062958573*x**3 - 4.42324588157734*x**2 - 1.73755734900681*x - 0.780513208628592

In [8]:
s, l = datalist_generator(funct, 15, 15, 28, 28)

In [9]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [10]:
from tensorflow.keras.models import load_model
model = load_model('Unsigned_Research_main.h5')

In [11]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
6/6 - 1s - loss: 4.4092e-09 - mean_absolute_error: 4.0868e-05 - val_loss: 1.7177e-09 - val_mean_absolute_error: 2.9306e-05 - 1s/epoch - 246ms/step
Epoch 2/30
6/6 - 0s - loss: 1.9485e-09 - mean_absolute_error: 2.7558e-05 - val_loss: 3.5129e-11 - val_mean_absolute_error: 4.1910e-06 - 65ms/epoch - 11ms/step
Epoch 3/30
6/6 - 0s - loss: 1.0661e-09 - mean_absolute_error: 2.0793e-05 - val_loss: 1.3943e-10 - val_mean_absolute_error: 8.3495e-06 - 58ms/epoch - 10ms/step
Epoch 4/30
6/6 - 0s - loss: 6.2231e-10 - mean_absolute_error: 1.6368e-05 - val_loss: 3.4105e-10 - val_mean_absolute_error: 1.3058e-05 - 53ms/epoch - 9ms/step
Epoch 5/30
6/6 - 0s - loss: 3.5398e-10 - mean_absolute_error: 1.1817e-05 - val_loss: 3.4395e-10 - val_mean_absolute_error: 1.3114e-05 - 72ms/epoch - 12ms/step
Epoch 6/30
6/6 - 0s - loss: 1.9035e-10 - mean_absolute_error: 8.7020e-06 - val_loss: 2.2592e-10 - val_mean_absolute_error: 1.0628e-05 - 59ms/epoch - 10ms/step
Epoch 7/30
6/6 - 0s - loss: 9.6589e-11 - mean_ab

In [12]:
model.save('Unsigned_Research_main.h5')

In [13]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

3/3 - 0s - loss: 2.7216e-17 - mean_absolute_error: 3.3822e-09 - 51ms/epoch - 17ms/step


In [14]:
funct = exp_generator(1, 4)

In [15]:
funct

3.72731749060969*x**4 + 1.91861767685701*x**3 + 0.189321916984779*x**2 - 2.347265527043*x + 0.691987276541325

In [16]:
s, l = datalist_generator(funct, 7, 7, 20, 20)

In [17]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [18]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
6/6 - 0s - loss: 4.7451e-17 - mean_absolute_error: 4.2586e-09 - val_loss: 1.4988e-17 - val_mean_absolute_error: 2.5243e-09 - 154ms/epoch - 26ms/step
Epoch 2/30
6/6 - 0s - loss: 3.0488e-17 - mean_absolute_error: 3.4158e-09 - val_loss: 5.6181e-17 - val_mean_absolute_error: 4.6843e-09 - 89ms/epoch - 15ms/step
Epoch 3/30
6/6 - 0s - loss: 2.1198e-17 - mean_absolute_error: 2.7490e-09 - val_loss: 1.4829e-17 - val_mean_absolute_error: 2.5034e-09 - 87ms/epoch - 14ms/step
Epoch 4/30
6/6 - 0s - loss: 1.3187e-17 - mean_absolute_error: 1.8629e-09 - val_loss: 1.2297e-17 - val_mean_absolute_error: 1.8150e-09 - 87ms/epoch - 15ms/step
Epoch 5/30
6/6 - 0s - loss: 9.2117e-18 - mean_absolute_error: 1.6983e-09 - val_loss: 1.3041e-17 - val_mean_absolute_error: 1.5695e-09 - 84ms/epoch - 14ms/step
Epoch 6/30
6/6 - 0s - loss: 7.9445e-18 - mean_absolute_error: 1.5164e-09 - val_loss: 1.5957e-17 - val_mean_absolute_error: 1.3919e-09 - 86ms/epoch - 14ms/step
Epoch 7/30
6/6 - 0s - loss: 8.3086e-18 - mean

In [19]:
model.save('Unsigned_Research_main.h5')

In [20]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

3/3 - 0s - loss: 4.6652e-18 - mean_absolute_error: 1.0800e-09 - 55ms/epoch - 18ms/step


### Once we are working with unsigned labels, the model is giving exceptional results for all types of univariate polynomial functions. So, the earlier issue is solved. In order to understand whether we can continue with this approach, we also test the model on the bivariate polynomial functions using a smaller no of samples.

In [6]:
funct = exp_generator(2, 2)

In [7]:
funct

4.32086999328798*x**2 - 1.64590280621048*x*y + 1.75877056634512*x + 0.31179056588644*y**2 + 4.03502610685158*y - 2.2291855911713

In [8]:
s, l = datalist_generator(funct, 1, 1, 14, 14)

In [9]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [10]:
for i in train_labels:
    print(i)

[5.34960333e-01 4.35623035e-04]
[1.44816727e+00 1.42463835e-03]
[0.42332068 0.00050381]
[4.78983446e-03 1.58465315e-05]
[7.27205604e-03 2.56557919e-05]
[2.79744058e-03 5.73223361e-06]
[0.10491997 0.00059741]
[5.99812275e-03 2.26723913e-05]
[3.99402128e-03 1.14151845e-05]
[5.57464032e-03 1.78973170e-05]
[4.45022538e-03 1.72710320e-05]
[9.16359307e-03 5.91462048e-05]
[9.73477368e-03 4.71439037e-05]
[3.14249494e-03 7.52050128e-06]
[0.34322201 0.00070627]
[3.71590452e-03 1.05262432e-05]
[0.38583756 0.00042233]
[1.14417812e+00 1.02185756e-03]
[4.88496347e-03 1.17174647e-05]
[0.01460116 0.00010043]
[0.7520007  0.00076235]
[1.03024922e-02 3.79161719e-05]
[0.03456576 0.00028746]
[0.10094175 0.00035305]
[8.31859861e-01 8.09446334e-04]
[0.19374521 0.00038788]
[1.38455136e-02 7.22025154e-05]
[3.46468039e-03 6.70403057e-06]
[0.0300154  0.00018115]
[5.75824066e-03 2.66265848e-05]
[0.09616602 0.00022544]
[1.10990965e-02 6.56295936e-05]
[3.48088347e-03 9.69199778e-06]
[0.03166544 0.00012651]
[0.01541

In [ ]:
for i in test_labels:
    print(i)

##### The bivariate polynomial functions have 2 curvature values.

In [12]:
from tensorflow.keras.models import load_model
model = load_model('Unsigned_Research_main.h5')

In [13]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
6/6 - 2s - loss: 0.0420 - mean_absolute_error: 0.0558 - val_loss: 0.0090 - val_mean_absolute_error: 0.0247 - 2s/epoch - 342ms/step
Epoch 2/30
6/6 - 0s - loss: 0.0417 - mean_absolute_error: 0.0546 - val_loss: 0.0089 - val_mean_absolute_error: 0.0237 - 101ms/epoch - 17ms/step
Epoch 3/30
6/6 - 0s - loss: 0.0415 - mean_absolute_error: 0.0539 - val_loss: 0.0088 - val_mean_absolute_error: 0.0230 - 82ms/epoch - 14ms/step
Epoch 4/30
6/6 - 0s - loss: 0.0412 - mean_absolute_error: 0.0536 - val_loss: 0.0087 - val_mean_absolute_error: 0.0228 - 86ms/epoch - 14ms/step
Epoch 5/30
6/6 - 0s - loss: 0.0410 - mean_absolute_error: 0.0535 - val_loss: 0.0086 - val_mean_absolute_error: 0.0230 - 85ms/epoch - 14ms/step
Epoch 6/30
6/6 - 0s - loss: 0.0409 - mean_absolute_error: 0.0535 - val_loss: 0.0086 - val_mean_absolute_error: 0.0230 - 84ms/epoch - 14ms/step
Epoch 7/30
6/6 - 0s - loss: 0.0407 - mean_absolute_error: 0.0536 - val_loss: 0.0085 - val_mean_absolute_error: 0.0231 - 86ms/epoch - 14ms/step

In [14]:
model.save('Unsigned_Research_main.h5')

In [15]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

3/3 - 0s - loss: 0.0677 - mean_absolute_error: 0.0856 - 46ms/epoch - 15ms/step


### Since, the model is now working on a different type of data, the results are comparatively worse. However, they are not significantly bad given we are working with a small dataset and also moving from one type of dataset to another.

In [16]:
funct = exp_generator(2, 3)

In [17]:
funct

2.47372831898284*x**3 + 2.65933512466766*x**2*y + 4.60571124014471*x**2 - 2.67107083583941*x*y**2 + 4.00245921680077*x*y - 1.01387410921967*x - 4.49247913802228*y**3 - 0.63492343236069*y**2 - 3.00606742962891*y + 4.25669800630308

In [18]:
s, l = datalist_generator(funct, 10, 10, 23, 23)

In [19]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [20]:
for i in train_labels:
    print(i)

[7.5223939e-03 1.5647880e-07]
[4.01730281e-02 6.80581543e-09]
[3.83653245e-02 4.85099572e-09]
[4.43061412e-02 1.29974921e-08]
[1.03456255e-03 1.58092929e-07]
[2.44228888e-02 1.69772182e-09]
[1.64537895e-02 2.22620749e-09]
[6.51183559e-02 5.39030197e-09]
[1.99021007e-02 1.56276166e-08]
[1.79776147e-02 5.50958751e-08]
[3.26538386e-03 1.48070950e-08]
[1.06088055e-02 4.01886583e-09]
[5.82206492e-02 7.02430137e-09]
[2.31162440e-03 9.83802568e-07]
[4.00876163e-02 1.38815818e-08]
[3.66093643e-02 3.84472749e-09]
[4.12587504e-02 3.76397034e-09]
[3.20674176e-02 2.37858041e-09]
[3.01160283e-02 1.90397535e-09]
[4.61972067e-03 3.16810763e-08]
[1.92733750e-02 5.20504065e-09]
[7.09018389e-03 5.61217521e-09]
[1.70686216e-02 2.73274744e-07]
[3.15979545e-02 1.84288586e-09]
[5.02137418e-02 3.20326667e-08]
[7.1391615e-03 9.7712050e-09]
[7.03723560e-03 1.84104222e-08]
[3.20615962e-02 1.38749044e-09]
[1.07938820e-02 3.97613806e-08]
[1.34067553e-02 2.58898399e-09]
[3.73021627e-02 7.15002342e-09]
[3.00693966e

In [ ]:
for i in test_labels:
    print(i)

In [22]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
6/6 - 0s - loss: 6310.7886 - mean_absolute_error: 19.9486 - val_loss: 1.3073e-04 - val_mean_absolute_error: 0.0071 - 147ms/epoch - 25ms/step
Epoch 2/30
6/6 - 0s - loss: 1.3045e-04 - mean_absolute_error: 0.0072 - val_loss: 1.2525e-04 - val_mean_absolute_error: 0.0074 - 85ms/epoch - 14ms/step
Epoch 3/30
6/6 - 0s - loss: 1.2880e-04 - mean_absolute_error: 0.0074 - val_loss: 1.2279e-04 - val_mean_absolute_error: 0.0075 - 83ms/epoch - 14ms/step
Epoch 4/30
6/6 - 0s - loss: 1.2817e-04 - mean_absolute_error: 0.0075 - val_loss: 1.2160e-04 - val_mean_absolute_error: 0.0076 - 82ms/epoch - 14ms/step
Epoch 5/30
6/6 - 0s - loss: 1.2790e-04 - mean_absolute_error: 0.0076 - val_loss: 1.2100e-04 - val_mean_absolute_error: 0.0076 - 82ms/epoch - 14ms/step
Epoch 6/30
6/6 - 0s - loss: 1.2789e-04 - mean_absolute_error: 0.0077 - val_loss: 1.2067e-04 - val_mean_absolute_error: 0.0077 - 83ms/epoch - 14ms/step
Epoch 7/30
6/6 - 0s - loss: 1.2779e-04 - mean_absolute_error: 0.0077 - val_loss: 1.2049e-04 -

In [23]:
model.save('Unsigned_Research_main.h5')

# Deleted, as the model was trained on a smaller dataset, and also, it was eventually found out that this architecture wasn't
# upto the mark for our purpose.

In [24]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

3/3 - 0s - loss: 1.7314e-04 - mean_absolute_error: 0.0089 - 49ms/epoch - 16ms/step


### Looking at these results, it seems that the model has potential to improve if trained on a larger dataset. Hence, we decided to continue working with unsigned labels and test the model on a larger dataset of different polynomial function values.

## Training and testing with a larger dataset with "unsigned" labels from scratch

In [6]:
funct = exp_generator(1, 2)

In [7]:
funct

-2.34930594962757*x**2 + 4.60161752198358*x - 4.05231083546138

In [8]:
s, l = datalist_generator(funct, 25, 25, 150, 150)

In [9]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [10]:
model = Sequential([
    Dense(units = 6, input_shape = (9,), activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 36, activation = 'relu'),
    Dense(units = 24, activation = 'relu'),
    Dense(units = 12, activation = 'relu'),
    Dense(units = 6, activation = 'relu'),
    Dense(units = 2)
])

In [11]:
model.compile(optimizer = Adam(learning_rate = 0.0001), loss = 'mean_squared_error', metrics = ['mean_absolute_error'])

In [12]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 3s - loss: 325167.1250 - mean_absolute_error: 324.9802 - val_loss: 4100.7373 - val_mean_absolute_error: 47.9998 - 3s/epoch - 8ms/step
Epoch 2/30
417/417 - 1s - loss: 1017.4055 - mean_absolute_error: 21.2924 - val_loss: 33.4311 - val_mean_absolute_error: 4.6396 - 1s/epoch - 3ms/step
Epoch 3/30
417/417 - 1s - loss: 7.2922 - mean_absolute_error: 2.0076 - val_loss: 2.6363 - val_mean_absolute_error: 1.3211 - 1s/epoch - 3ms/step
Epoch 4/30
417/417 - 1s - loss: 2.4704 - mean_absolute_error: 1.2972 - val_loss: 2.3831 - val_mean_absolute_error: 1.2871 - 1s/epoch - 3ms/step
Epoch 5/30
417/417 - 1s - loss: 2.2696 - mean_absolute_error: 1.2723 - val_loss: 2.2017 - val_mean_absolute_error: 1.2675 - 1s/epoch - 3ms/step
Epoch 6/30
417/417 - 1s - loss: 2.1280 - mean_absolute_error: 1.2531 - val_loss: 2.0802 - val_mean_absolute_error: 1.2445 - 1s/epoch - 3ms/step
Epoch 7/30
417/417 - 1s - loss: 2.0187 - mean_absolute_error: 1.2344 - val_loss: 1.9738 - val_mean_absolute_error: 1.228

In [13]:
model.save('Larger_Unsigned_Research_main.h5')

In [14]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

239/239 - 0s - loss: 8.8318e-04 - mean_absolute_error: 0.0184 - 400ms/epoch - 2ms/step


In [6]:
funct = exp_generator(1, 3)

In [7]:
funct

1.50598542228715*x**3 - 0.182709651986466*x**2 - 3.06113867322723*x - 4.71392323054118

In [8]:
s, l = datalist_generator(funct, 1, 1, 126, 126)

In [9]:
train_samples, test_samples, train_labels, test_labels = train_test_split(s, l, test_size = 0.3)

In [10]:
from tensorflow.keras.models import load_model
model = load_model('Larger_Unsigned_Research_main.h5')

In [11]:
model.fit(train_samples, train_labels, validation_split = 0.25, batch_size = 20, epochs = 30, shuffle = True, verbose = 2)

Epoch 1/30
417/417 - 3s - loss: 87976760.0000 - mean_absolute_error: 1465.3672 - val_loss: 189004.2031 - val_mean_absolute_error: 346.7651 - 3s/epoch - 7ms/step
Epoch 2/30
417/417 - 1s - loss: 191192.6875 - mean_absolute_error: 343.1601 - val_loss: 191766.9688 - val_mean_absolute_error: 342.9899 - 1s/epoch - 3ms/step
Epoch 3/30
417/417 - 1s - loss: 88586.5781 - mean_absolute_error: 211.3429 - val_loss: 59405.8359 - val_mean_absolute_error: 167.7213 - 1s/epoch - 3ms/step
Epoch 4/30
417/417 - 1s - loss: 58516.5000 - mean_absolute_error: 163.9048 - val_loss: 65895.4844 - val_mean_absolute_error: 181.9566 - 1s/epoch - 3ms/step
Epoch 5/30
417/417 - 1s - loss: 3287.6575 - mean_absolute_error: 23.4105 - val_loss: 83.6286 - val_mean_absolute_error: 7.7497 - 1s/epoch - 3ms/step
Epoch 6/30
417/417 - 1s - loss: 67.2643 - mean_absolute_error: 6.6997 - val_loss: 56.5003 - val_mean_absolute_error: 5.9387 - 1s/epoch - 3ms/step
Epoch 7/30
417/417 - 1s - loss: 52.5412 - mean_absolute_error: 5.6198 - va

In [ ]:
model.save('Larger_Unsigned_Research_main.h5')

# Deleted, as this model failed to give an acceptable performance.

In [ ]:
result = model.evaluate(test_samples, test_labels, batch_size = 20, verbose = 2)

### These results don't look promising at all, indicating the model we are working with isn't up to the mark for our purpose. This research is continued as I pursued my Master's, which is documented in another notebook.